# Joint regional-operator validation

Diagonal self-feedback and low-rank coupling are fitted jointly. Cross-state transfer uses one shared normalization, and time reversal uses episode-wise linear detrending to avoid imposing a direction through causal filtering.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from yumeow_plot import get_fig

ROOT = Path('..').resolve()
RUN = ROOT / 'logs/scripts/regional_operator_validation/initial'
OUT = ROOT / 'figures/261027区域算子联合验证'
OUT.mkdir(parents=True, exist_ok=True)
joint = pd.read_csv(RUN / 'joint_rank_summary.csv')
transfer = pd.read_csv(RUN / 'cross_state_transfer.csv')
reversal = pd.read_csv(RUN / 'time_reversal.csv')

In [2]:
colors = {'Awake': '#cc5544', 'NREM': '#4477aa', 'REM': '#aa55aa'}
fig, axes = get_fig(RN=1, CN=3, AW=4, AH=4, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(axes)
ax = next(ax_loader)
for state in ['Awake', 'NREM']:
    part = joint[joint.state == state]
    ax.plot(part['rank'], part.dx_r2_flat, marker='o', color=colors[state], label=state)
ax.set(xscale='log', xlabel='Coupling rank', ylabel=r'Held-out $\Delta x$ $R^2$', title='Joint diagonal + low-rank')
ax.set_xticks([1, 2, 3, 5, 8, 12, 25], [1, 2, 3, 5, 8, 12, 25])
ax.legend(frameon=False)
ax = next(ax_loader)
states = ['Awake', 'NREM', 'REM']
matrix = transfer[transfer['rank'] == 25].pivot(index='source_state', columns='target_state', values='dx_r2_flat').loc[['Awake', 'NREM', 'REM', 'Pooled'], states]
image = ax.imshow(matrix, cmap='RdBu_r', vmin=-.25, vmax=.25, aspect='auto')
ax.set_xticks(range(3), states, rotation=35, ha='right')
ax.set_yticks(range(4), matrix.index)
ax.set(xlabel='Test state', ylabel='Training state', title='Cross-state transfer')
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, f'{matrix.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
ax = next(ax_loader)
part = reversal[reversal['rank'] == 25]
x = np.arange(3)
for offset, direction, hatch in [(-.18, 'forward', ''), (.18, 'reversed', '//')]:
    values = part[part.direction == direction].set_index('state').loc[states].dx_r2_flat
    ax.bar(x + offset, values, width=.34, color=[colors[s] for s in states], hatch=hatch, alpha=.9, label=direction)
ax.set_xticks(x, states)
ax.set(ylabel=r'Held-out $\Delta x$ $R^2$', title='Time-reversal control')
ax.legend(frameon=False)
fig.savefig(OUT / 'operator_validation.png')
fig.savefig(OUT / 'operator_validation.svg')
fig.savefig(OUT / 'operator_validation.pdf')
plt.show()

## Interpretation

Joint diagonal plus rank-5 fitting retains nearly all full-operator performance: Awake 0.075 versus 0.080 and NREM 0.172 versus 0.175. NREM therefore admits a particularly compact predictive operator.

Transfer is asymmetric. An Awake-trained full operator retains R² 0.131 on NREM, whereas an NREM-trained operator is negative on Awake. The pooled model is close to the Awake-specific model on Awake but loses substantial NREM performance (0.139 versus 0.176). NREM needs state-specific modulation beyond the dominant shared structure.

Time reversal is decisive for interpretation: reversed performance is comparable and is higher in Awake and REM. These lagged operators capture predictable covariance but do not yet establish a physical arrow of influence. Directional or effective-connectivity claims require asymmetric events, external perturbations, or additional surrogate evidence.